# Ingestion Logs Report
This notebook summarises ingestion audit logs: timings, errors, and key metrics.

In [ ]:
# Load and parse audit JSONL
import os, json, pathlib, statistics
from collections import Counter, defaultdict
from datetime import datetime

workspace_root = pathlib.Path(__file__).resolve().parents[1] if '__file__' in globals() else pathlib.Path.cwd().parents[0]
audit_path = workspace_root / 'logs' / 'ingest_audit.jsonl'

events = []
if audit_path.exists():
    with open(audit_path, 'r') as f:
        for line in f:
            try:
                events.append(json.loads(line))
            except Exception:
                pass
else:
    print(f'No audit log found at {audit_path}')

print('Number of events:', len(events))

In [ ]:
# Summarise event counts and basic metrics
event_counts = Counter(e.get('event') for e in events)

# Handle both regular ingest (done_file) and academic ingest (document_processed) events
total_files = event_counts.get('start_file', 0) + event_counts.get('start', 0)
completed_files = event_counts.get('done_file', 0) + event_counts.get('document_processed', 0)
skipped_files = event_counts.get('skip', 0)
errors = [e for e in events if e.get('event') in ('error','stage_error','chunk_validation_error','chunk_structural_validation_error','document_processing_failed')]

print('Event counts:', dict(event_counts))
print('Total files:', total_files, 'Completed:', completed_files, 'Skipped:', skipped_files, 'Errors:', len(errors))

In [ ]:
# Debug: Check what events are actually in the log
print('All unique event types in audit log:')
all_event_types = sorted(set(e.get('event') for e in events))
print(all_event_types)
print()

print('Events containing "error" in name:')
error_containing_events = [e for e in all_event_types if 'error' in e.lower()]
print(error_containing_events)
print()

print('Count of each error-related event:')
for evt in error_containing_events:
    count = sum(1 for e in events if e.get('event') == evt)
    print(f'  {evt}: {count}')

In [ ]:
# Timing summaries from done_file events (using *_seconds fields)
from collections import defaultdict

# Collect both done_file (regular ingest) and document_processed (academic ingest) events
done_file_events = [e for e in events if e.get('event') == 'done_file']
doc_processed_events = [e for e in events if e.get('event') == 'document_processed']

# Combine both event types
done = done_file_events + doc_processed_events

# Overall durations
durations = [float(e.get('duration_seconds', 0) or 0) for e in done]

# Known phase keys and friendly labels (supporting both event types)
phase_key_map = {
    # Regular ingest fields
    'parse_seconds': 'parse',
    'preprocess_seconds': 'preprocess',
    'chunk_seconds': 'chunk',
    'idempotency_seconds': 'idempotency',
    'storage_seconds': 'storage',
    # Academic ingest fields (map to similar phases)
    'preprocess_time': 'preprocess',
    'ingest_time': 'ingest',
}

phase_totals = defaultdict(float)
phase_counts = defaultdict(int)

for e in done:
    for key, label in phase_key_map.items():
        val = e.get(key)
        if val is not None:
            try:
                phase_totals[label] += float(val)
                phase_counts[label] += 1
            except Exception:
                pass

# Helper
def summarise(lst):
    if not lst:
        return {'count': 0, 'avg': 0, 'min': 0, 'max': 0}
    return {'count': len(lst), 'avg': sum(lst)/len(lst), 'min': min(lst), 'max': max(lst)}

print('Duration summary:', summarise(durations))

# Print totals and averages per phase
print('Phase totals:', dict(phase_totals))
phase_avgs = {k: (phase_totals[k] / phase_counts[k]) if phase_counts[k] else 0 for k in phase_totals}
print('Phase averages:', phase_avgs)
print('Phase counts:', dict(phase_counts))

In [ ]:
# Visualisation: Stacked bar chart of per-phase time by document with tooltips and source_category filter
import os
import math
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure events are loaded if this cell is run first
if 'events' not in globals():
    workspace_root = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path.cwd().parents[0]
    audit_path = workspace_root / 'logs' / 'ingest_audit.jsonl'
    events = []
    if audit_path.exists():
        with open(audit_path, 'r') as f:
            for line in f:
                try:
                    events.append(json.loads(line))
                except Exception:
                    pass
    else:
        print(f'No audit log found at {audit_path}')

# Reuse done and phase_key_map from previous cell if present; otherwise rebuild
try:
    done
    phase_key_map
except NameError:
    done_file_events = [e for e in events if e.get('event') == 'done_file']
    doc_processed_events = [e for e in events if e.get('event') == 'document_processed']
    done = done_file_events + doc_processed_events
    
    phase_key_map = {
        # Regular ingest fields
        'parse_seconds': 'parse',
        'preprocess_seconds': 'preprocess',
        'chunk_seconds': 'chunk',
        'idempotency_seconds': 'idempotency',
        'storage_seconds': 'storage',
        # Academic ingest fields
        'preprocess_time': 'preprocess',
        'ingest_time': 'ingest',
    }

# Map doc_id -> source_category from audit events (if available)
cat_by_doc = {}
for ev in events:
    if ev.get('event') == 'source_category_detected':
        did = ev.get('doc_id')
        cat = ev.get('category')
        if did and cat:
            cat_by_doc[did] = cat

# Build records per doc - supporting both event types
phase_labels = ['parse', 'preprocess', 'chunk', 'idempotency', 'storage', 'ingest']
records = []
for e in done:
    rec = {'doc_id': e.get('doc_id', 'UNKNOWN')}
    
    # Initialise all phases to 0
    for label in phase_labels:
        rec[label] = 0.0
    
    # Map event fields to phases
    for raw, label in phase_key_map.items():
        val = e.get(raw)
        try:
            if val is not None:
                rec[label] += float(val)  # += to handle duplicate mappings (e.g., preprocess)
        except Exception:
            pass
    
    rec['total'] = sum(rec[label] for label in phase_labels)
    rec['source_category'] = cat_by_doc.get(rec['doc_id'], 'Unknown')
    records.append(rec)

if not records:
    print('No done_file or document_processed events found; cannot build visualisation.')
else:
    df = pd.DataFrame(records)

    # Optional filter by source_category via env or local variable
    if 'filter_source_category' not in locals():
        filter_source_category = os.getenv('FILTER_SOURCE_CATEGORY')

    if filter_source_category:
        before = len(df)
        df = df[df['source_category'] == filter_source_category]
        print(f"Applied source_category filter: '{filter_source_category}' ({len(df)}/{before} docs)")
    else:
        print('No source_category filter applied')

    if df.empty:
        print('No documents to display after filtering.')
    else:
        # Sort by total duration desc and limit to top N for readability
        TOP_N = 20
        df = df.sort_values('total', ascending=False).head(TOP_N)

        # Try interactive Plotly first; fallback to matplotlib
        try:
            import plotly.express as px
            width_px = max(1000, min(2000, len(df) * 90))  # widen to better fit long doc_ids
            height_px = 750
            fig = px.bar(
                df,
                x='doc_id',
                y=['parse', 'preprocess', 'chunk', 'idempotency', 'storage', 'ingest'],
                barmode='stack',
                title='Per-Document Phase Durations (Top by total)'
            )
            # Attach customdata: total and source_category for hover
            custom = np.stack([df['total'].values, df['source_category'].values], axis=-1)
            for tr in fig.data:
                tr.customdata = custom
                tr.hovertemplate = (
                    'Phase=%{fullData.name}<br>'
                    'Doc=%{x}<br>'
                    'Seconds=%{y:.3f}<br>'
                    'Total=%{customdata[0]:.3f}<br>'
                    'Category=%{customdata[1]}'
                    '<extra></extra>'
                )
            fig.update_layout(
                xaxis={'tickangle': 45, 'automargin': True},
                margin={'l': 80, 'r': 80, 't': 80, 'b': 120},
                bargap=0.15,
                width=width_px,
                height=height_px,
            )
            fig.show()
        except Exception as ex:
            print(f"Plotly unavailable or failed ({ex}); falling back to matplotlib.")
            width = max(12, min(30, len(df) * 0.8))  # widen to allow longer labels
            plt.figure(figsize=(width, 10))
            x = range(len(df))
            bottoms = [0.0] * len(df)

            # Consistent phase order
            ordered_phases = ['parse', 'preprocess', 'chunk', 'idempotency', 'storage', 'ingest']
            colours = {
                'parse': '#6baed6',
                'preprocess': '#74c476',
                'chunk': '#fd8d3c',
                'idempotency': '#9e9ac8',
                'storage': '#fdae6b',
                'ingest': '#fc8d62',
            }

            for phase in ordered_phases:
                values = df[phase].tolist()
                if sum(values) > 0:  # Only plot if phase has data
                    plt.bar(x, values, bottom=bottoms, label=phase, color=colours.get(phase))
                    bottoms = [b + v for b, v in zip(bottoms, values)]

            # Labels and ticks
            doc_labels = df['doc_id'].tolist()
            def truncate(s, n=40):
                return s if len(str(s)) <= n else str(s)[:n-3] + '...'
            plt.xticks(x, [truncate(s) for s in doc_labels], rotation=45, ha='right')
            plt.ylabel('Seconds')
            plt.title('Per-Document Phase Durations (Top by total)')
            plt.legend(title='Phase', bbox_to_anchor=(1.02, 1), loc='upper left')
            plt.tight_layout()
            plt.show()

        # Print a compact summary table under the chart
        summary_cols = ['doc_id', 'source_category', 'parse', 'preprocess', 'chunk', 'idempotency', 'storage', 'ingest', 'total']
        # Only show columns that have data
        active_cols = ['doc_id', 'source_category'] + [c for c in ['parse', 'preprocess', 'chunk', 'idempotency', 'storage', 'ingest'] if df[c].sum() > 0] + ['total']
        print('\nTop documents by total duration:')
        print(df[active_cols].to_string(index=False))

In [ ]:
# Debug: Check what fields are in done_file and document_processed events
print('Checking event structure:')
done_file_events = [e for e in events if e.get('event') == 'done_file']
doc_processed_events = [e for e in events if e.get('event') == 'document_processed']

print(f'Total done_file events: {len(done_file_events)}')
print(f'Total document_processed events: {len(doc_processed_events)}')
print()

if done_file_events:
    first_done = done_file_events[0]
    print('Keys in first done_file event:')
    print(list(first_done.keys()))
    print()
    
    print('Fields that might contain timing/phase data:')
    for key in first_done.keys():
        if any(x in key.lower() for x in ['phase', 'time', 'duration', 'ingest', 'preprocess', 'chunk', 'validate', 'store']):
            print(f'  {key}: {first_done[key]}')
    print()

if doc_processed_events:
    first_doc = doc_processed_events[0]
    print('Keys in first document_processed event:')
    print(list(first_doc.keys()))
    print()
    
    print('Fields that might contain timing/phase data:')
    for key in first_doc.keys():
        if any(x in key.lower() for x in ['phase', 'time', 'duration', 'ingest', 'preprocess', 'chunk', 'validate', 'store']):
            print(f'  {key}: {first_doc[key]}')

In [ ]:
# Error breakdown
print(f'Total errors found: {len(errors)}')
print(f'Event types in errors: {set(e.get("event") for e in errors)}')
print()

if errors:
    # Get error types - handle both error_type field and validation errors
    error_types = []
    for e in errors:
        et = e.get('error_type')
        if et:
            error_types.append(et)
        elif e.get('event') == 'chunk_validation_error' and e.get('errors'):
            error_types.append(e.get('errors')[0].get('type', 'validation_error'))
        else:
            error_types.append(e.get('event', 'unknown'))
    
    err_types = Counter(error_types)
    top_errors = sorted(err_types.items(), key=lambda x: x[1], reverse=True)[:10]
    print('Top error types:', top_errors)
    print()
    
    print('Sample errors (up to 5):')
    for e in errors[:5]:
        if e.get('event') == 'chunk_validation_error':
            error_info = e.get('errors')[0] if e.get('errors') else {}
            print(f"  Chunk validation error: {e.get('chunk_id')}")
            print(f"    Type: {error_info.get('type')}")
            print(f"    Message: {error_info.get('msg')}")
        elif e.get('event') == 'chunk_structural_validation_error':
            print(f"  Structural validation error: {e.get('chunk_id')}")
        else:
            print({k: e.get(k) for k in ('timestamp','event','stage','path','doc_id','error_type','error')})
        print()
else:
    print('✗ No errors found in audit log')

In [ ]:
# Language detection summary from ChromaDB
from pathlib import Path
import chromadb
from collections import Counter

# Get workspace root and Chroma path
workspace_root = pathlib.Path(__file__).resolve().parents[1] if '__file__' in globals() else pathlib.Path.cwd().parents[0]
chroma_path = workspace_root / 'rag_data' 

if chroma_path.exists():
    try:
        # Initialise Chroma client
        client = chromadb.PersistentClient(path=str(chroma_path))
        
        # Get all collections
        collections = client.list_collections()
        print(f"Found {len(collections)} collection(s) in ChromaDB")
        print()
        
        if collections:
            for collection in collections:
                print(f"Collection: {collection.name}")
                
                # Get all metadata from collection
                try:
                    all_docs = collection.get(include=['metadatas'])
                    
                    if all_docs and all_docs['metadatas']:
                        # Extract language field from metadatas
                        languages = []
                        for metadata in all_docs['metadatas']:
                            if metadata and 'language' in metadata:
                                lang = metadata['language']
                                if lang and lang.strip():
                                    languages.append(lang)
                        
                        if languages:
                            # Count occurrences
                            lang_counts = Counter(languages)
                            total = len(languages)
                            
                            print(f"  Total chunks with language metadata: {total}")
                            print(f"  Unique languages detected: {len(lang_counts)}")
                            print()
                            print("  Language breakdown:")
                            for lang, count in sorted(lang_counts.items(), key=lambda x: x[1], reverse=True):
                                pct = (count / total * 100) if total > 0 else 0
                                print(f"    {lang:20s}: {count:6d} ({pct:5.1f}%)")
                        else:
                            print(f"  No language metadata found in {len(all_docs['metadatas'])} documents")
                    else:
                        print(f"  No documents found in collection")
                except Exception as e:
                    print(f"  Error querying collection: {e}")
                
                print()
        else:
            print("No collections found in ChromaDB")
            
    except Exception as e:
        print(f"Error connecting to ChromaDB: {e}")
else:
    print(f"ChromaDB path not found: {chroma_path}")


## Truncation Analysis

Assess the degree of chunk truncation during embedding generation.

In [ ]:
# Extract truncation statistics from document_health events
truncation_data = []

for log_entry in events:
    if log_entry.get("event") == "document_health":
        doc_id = log_entry.get("doc_id", "unknown")
        health = log_entry.get("health", {})
        
        truncation_data.append({
            "doc_id": doc_id,
            "total_chunks": health.get("total_chunks", 0),
            "truncated_chunks": health.get("truncated_chunks", 0),
            "truncation_ratio": health.get("chunk_truncation_ratio", 0.0),
            "truncation_loss_avg_pct": health.get("chunk_truncation_loss_avg_pct", 0.0),
            "truncation_chars_lost": health.get("chunk_truncation_chars_lost", 0),
        })

if truncation_data:
    df_truncation = pd.DataFrame(truncation_data)
    
    # Overall statistics
    total_docs = len(df_truncation)
    total_chunks_all = df_truncation["total_chunks"].sum()
    total_truncated = df_truncation["truncated_chunks"].sum()
    overall_truncation_ratio = total_truncated / total_chunks_all if total_chunks_all > 0 else 0.0
    total_chars_lost = df_truncation["truncation_chars_lost"].sum()
    
    # Documents with truncation
    docs_with_truncation = df_truncation[df_truncation["truncated_chunks"] > 0]
    docs_with_truncation_count = len(docs_with_truncation)
    docs_with_truncation_pct = (docs_with_truncation_count / total_docs) * 100 if total_docs > 0 else 0.0
    
    # Average loss percentage (only for documents that had truncation)
    avg_loss_pct = docs_with_truncation["truncation_loss_avg_pct"].mean() if len(docs_with_truncation) > 0 else 0.0
    
    print("=" * 70)
    print("TRUNCATION SUMMARY")
    print("=" * 70)
    print(f"Total documents:              {total_docs}")
    print(f"Documents with truncation:    {docs_with_truncation_count} ({docs_with_truncation_pct:.1f}%)")
    print(f"Total chunks:                 {total_chunks_all:,}")
    print(f"Truncated chunks:             {total_truncated:,}")
    print(f"Overall truncation ratio:     {overall_truncation_ratio:.2%}")
    print(f"Total characters lost:        {total_chars_lost:,}")
    print(f"Avg loss per truncated chunk: {avg_loss_pct:.2f}%")
    print()
    
    # Top 10 documents by truncation ratio
    if len(docs_with_truncation) > 0:
        top_truncation = docs_with_truncation.nlargest(10, "truncation_ratio")
        print("\nTop 10 Documents by Truncation Ratio:")
        print("-" * 70)
        for idx, row in top_truncation.iterrows():
            print(f"{row['doc_id'][:60]}")
            print(f"  Chunks: {row['total_chunks']:>5} | Truncated: {row['truncated_chunks']:>5} | "
                  f"Ratio: {row['truncation_ratio']:>6.2%} | Avg Loss: {row['truncation_loss_avg_pct']:>5.2f}%")
else:
    print("No truncation data found in audit logs.")

## Recommendations
- Increase `MAX_WORKERS` if average duration is high and errors are low.
- Review top error types; fix frequent failures before full runs.
- Check phase totals: if `validate` dominates, enable `ENABLE_CHUNK_HEURISTIC_SKIP`.